# VateCon — Frontend Tests
Tests d'intégration frontend : vérification des routes, du proxy Nginx, du rendu des pages et des flux WebSocket depuis le point de vue du client.

> **Prérequis** : `docker compose up -d` doit être lancé (frontend sur port 3000).

In [ ]:
%pip install requests websocket-client colorama --quiet

In [ ]:
import requests
import json
import uuid
import threading
import websocket
from colorama import Fore, Style, init
init(autoreset=True)

FRONTEND_URL = "http://localhost:3000"
WS_URL       = "ws://localhost:3000"

results = []

def test(name, passed, detail=""):
    status = f"{Fore.GREEN}✓ PASS{Style.RESET_ALL}" if passed else f"{Fore.RED}✗ FAIL{Style.RESET_ALL}"
    results.append({"name": name, "passed": passed, "detail": detail})
    print(f"{status}  {name}" + (f"  →  {detail}" if detail else ""))

print("Frontend URL:", FRONTEND_URL)
print("─" * 60)

## 1. Pages HTML — Rendu et routing SPA

In [ ]:
# Page principale
r = requests.get(f"{FRONTEND_URL}/")
test("GET / — status 200", r.status_code == 200)
test("GET / — Content-Type HTML", "text/html" in r.headers.get("Content-Type", ""))
test("GET / — contient <div id=root>", '<div id="root">' in r.text)
test("GET / — titre VateCon", "VateCon" in r.text)

# Routes SPA (React Router) — Nginx doit renvoyer index.html
for route in ["/admin", "/knowledge", "/some/unknown/route"]:
    r = requests.get(f"{FRONTEND_URL}{route}")
    test(f"SPA fallback {route} — status 200", r.status_code == 200)
    test(f"SPA fallback {route} — retourne index.html", '<div id="root">' in r.text)

## 2. Assets statiques — JS/CSS chargés

In [ ]:
from html.parser import HTMLParser

class AssetParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.scripts = []
        self.styles  = []
    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "script" and attrs.get("src"):
            self.scripts.append(attrs["src"])
        if tag == "link" and attrs.get("rel") == "stylesheet":
            self.styles.append(attrs.get("href", ""))

html = requests.get(f"{FRONTEND_URL}/").text
parser = AssetParser()
parser.feed(html)

test("Assets — au moins 1 fichier JS", len(parser.scripts) > 0, f"{len(parser.scripts)} script(s)")

for src in parser.scripts[:3]:
    url = f"{FRONTEND_URL}{src}" if src.startswith("/") else src
    r = requests.get(url)
    test(f"Asset JS chargé — {src[:50]}", r.status_code == 200, f"{len(r.content)} bytes")

for href in parser.styles[:2]:
    if href:
        url = f"{FRONTEND_URL}{href}" if href.startswith("/") else href
        r = requests.get(url)
        test(f"Asset CSS chargé — {href[:50]}", r.status_code == 200, f"{len(r.content)} bytes")

## 3. Proxy API — Nginx → Backend

In [ ]:
# Le frontend Nginx proxifie /api/* vers backend:8000
r = requests.get(f"{FRONTEND_URL}/api/health")
test("Proxy /api/health — status 200", r.status_code == 200)
test("Proxy /api/health — réponse JSON valide", r.headers.get("Content-Type", "").startswith("application/json"))
test("Proxy /api/health — status ok", r.json().get("status") == "ok", str(r.json()))

r = requests.get(f"{FRONTEND_URL}/api/admin/stats")
test("Proxy /api/admin/stats — status 200", r.status_code == 200)
stats = r.json()
test("Proxy /api/admin/stats — données reçues", "total_conversations" in stats)

r = requests.get(f"{FRONTEND_URL}/api/knowledge/documents")
test("Proxy /api/knowledge/documents — status 200", r.status_code == 200)
test("Proxy /api/knowledge/documents — retourne liste", isinstance(r.json(), list))

## 4. Proxy FAQ — POST via frontend

In [ ]:
faq_payload = {
    "entries": [
        {"question": "Frontend test question?", "answer": "This is a test answer from the frontend notebook.", "category": "test"}
    ]
}

r = requests.post(
    f"{FRONTEND_URL}/api/knowledge/faq",
    json=faq_payload,
    headers={"Content-Type": "application/json"}
)
test("POST via proxy /api/knowledge/faq — status 200", r.status_code == 200)
test("POST FAQ — entries_added == 1", r.json().get("entries_added") == 1, str(r.json()))

## 5. Proxy WebSocket — Chat via Nginx

In [ ]:
session_id = str(uuid.uuid4())
ws_events   = []
ws_ready    = threading.Event()
ws_answered = threading.Event()

def on_msg(ws, msg):
    data = json.loads(msg)
    ws_events.append(data)
    t = data.get("type")
    print(f"  ← WS [{t}]", end="")
    if t == "connected":
        ws_ready.set()
        print(f" conv_id={data.get('conversation_id','')[:8]}...")
    elif t == "message":
        ws_answered.set()
        print(f" confidence={data.get('confidence')}")
    else:
        print()

def on_err(ws, err):
    print(f"{Fore.RED}  WS error: {err}")
    ws_ready.set(); ws_answered.set()

ws_client = websocket.WebSocketApp(
    f"{WS_URL}/ws/chat/{session_id}",
    on_message=on_msg,
    on_error=on_err,
)
threading.Thread(target=ws_client.run_forever, daemon=True).start()

connected = ws_ready.wait(timeout=5)
test("WebSocket via Nginx — connexion établie", connected)
test("WebSocket — event connected reçu", any(e.get("type") == "connected" for e in ws_events))

In [ ]:
# Envoyer un message via le proxy WebSocket Nginx
ws_answered.clear()
ws_client.send(json.dumps({"message": "Do you offer a free trial?"}))
print("  → Message envoyé via proxy WebSocket")

got_answer = ws_answered.wait(timeout=30)
test("WebSocket via proxy — réponse IA reçue", got_answer)

ai_msgs = [e for e in ws_events if e.get("type") == "message"]
if ai_msgs:
    m = ai_msgs[-1]
    test("Réponse IA via proxy — contenu non vide", len(m.get("content", "")) > 10, f"{len(m.get('content',''))} chars")
    test("Réponse IA via proxy — confidence dans [0,1]", 0 <= m.get("confidence", -1) <= 1, str(m.get("confidence")))
    test("Réponse IA via proxy — sources présentes", isinstance(m.get("sources"), list))
    print(f"\n  Réponse: {m.get('content','')[:300]}...")

ws_client.close()

## 6. Sécurité — Headers HTTP

In [ ]:
r = requests.get(f"{FRONTEND_URL}/")
headers = r.headers
print("Headers reçus:")
for k, v in headers.items():
    print(f"  {k}: {v}")

test("Header — Server présent", "server" in {k.lower() for k in headers})
test("Header — Content-Type correct", "text/html" in headers.get("Content-Type", ""))

# Vérifier que les routes inconnues retournent 200 (SPA) et non 404
r = requests.get(f"{FRONTEND_URL}/page-qui-nexiste-pas")
test("Route inconnue — SPA 200 (pas de 404)", r.status_code == 200)

## 7. Performance — Temps de réponse

In [ ]:
import time

endpoints = [
    ("GET /",                          f"{FRONTEND_URL}/"),
    ("GET /api/health",                f"{FRONTEND_URL}/api/health"),
    ("GET /api/admin/stats",           f"{FRONTEND_URL}/api/admin/stats"),
    ("GET /api/knowledge/documents",   f"{FRONTEND_URL}/api/knowledge/documents"),
]

print(f"{'Endpoint':<40} {'Durée':>10}  {'Status':>8}")
print("─" * 62)
for label, url in endpoints:
    start = time.time()
    r = requests.get(url)
    elapsed = (time.time() - start) * 1000
    ok = elapsed < 2000
    color = Fore.GREEN if ok else Fore.YELLOW
    print(f"{label:<40} {color}{elapsed:>8.0f}ms{Style.RESET_ALL}  {r.status_code:>8}")
    test(f"Performance {label} < 2000ms", ok, f"{elapsed:.0f}ms")

## 8. Résumé final

In [ ]:
passed = sum(1 for r in results if r["passed"])
failed = sum(1 for r in results if not r["passed"])
total  = len(results)

print("═" * 60)
print(f"RÉSULTATS : {Fore.GREEN}{passed} passés{Style.RESET_ALL} / {Fore.RED}{failed} échoués{Style.RESET_ALL} / {total} total")
print(f"Score     : {round(passed/total*100)}%" if total else "")
print("═" * 60)

if failed:
    print(f"\n{Fore.RED}Tests échoués :")
    for r in results:
        if not r["passed"]:
            print(f"  ✗ {r['name']}" + (f" → {r['detail']}" if r['detail'] else ""))